# Unit 12 — Binary Trees & Traversals

A school record system may need to search for one student or score every record without scanning unrelated branches. A **binary tree** gives each record at most two child positions, so one recursive walking pattern can search or score the whole structure fast.

## Lesson 1 — Store a Binary Tree Without Classes

Number the nodes from `0` through `N - 1`. Three parallel arrays store the value and two child indices of every node: `val[i]`, `left[i]`, and `right[i]`. A separate `root` index says where the tree begins.

The sentinel `-1` means "no child." A recursive walk MUST test `node == -1` before indexing an array. Python allows `arr[-1]` and silently reads the last element, so forgetting this base case causes a wrong answer instead of a crash. Use equality, `node == -1`; do not write `node is -1`, because identity is not the guaranteed way to compare integer values.

This input describes a root worth `8`, with children worth `3` and `10`:

```text
3 0
8 1 2
3 -1 -1
10 -1 -1
```

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    root = int(tokens[1])
    val = []
    left = []
    right = []
    position = 2
    i = 0
    while i < n:
        val.append(int(tokens[position]))
        left.append(int(tokens[position + 1]))
        right.append(int(tokens[position + 2]))
        position = position + 3
        i = i + 1

    def preorder(node):
        if node == -1:
            return ""
        text = str(val[node])
        left_text = preorder(left[node])
        right_text = preorder(right[node])
        if left_text != "":
            text = text + " " + left_text
        if right_text != "":
            text = text + " " + right_text
        return text

    return preorder(root)

assert solve("3 0 8 1 2 3 -1 -1 10 -1 -1") == "8 3 10"

## Three Recursive Traversal Orders

A traversal visits every reachable node exactly once. Only the position of "visit this node" changes:

- **Pre-order:** node, left subtree, right subtree.
- **In-order:** left subtree, node, right subtree.
- **Post-order:** left subtree, right subtree, node.

On the same non-symmetric tree, the three sequences differ. Build a sequence string with `+`; do not rely on a shortcut that ignores the tree's shape.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    root = int(tokens[1])
    val = []
    left = []
    right = []
    position = 2
    i = 0
    while i < n:
        val.append(int(tokens[position]))
        left.append(int(tokens[position + 1]))
        right.append(int(tokens[position + 2]))
        position = position + 3
        i = i + 1

    def add(first, second):
        if first == "":
            return second
        if second == "":
            return first
        return first + " " + second

    def preorder(node):
        if node == -1:
            return ""
        text = add(str(val[node]), preorder(left[node]))
        return add(text, preorder(right[node]))

    def inorder(node):
        if node == -1:
            return ""
        text = add(inorder(left[node]), str(val[node]))
        return add(text, inorder(right[node]))

    def postorder(node):
        if node == -1:
            return ""
        text = add(postorder(left[node]), postorder(right[node]))
        return add(text, str(val[node]))

    return preorder(root) + "\n" + inorder(root) + "\n" + postorder(root)

tree = "5 0 8 1 2 3 3 -1 10 -1 4 1 -1 -1 14 -1 -1"
assert solve(tree) == "8 3 1 10 14\n1 3 8 10 14\n1 3 14 10 8"

## Recursive Aggregation

A traversal can return a score instead of a sequence. Define the **height** as the number of nodes on the longest root-to-leaf path, so an empty subtree has height `0` and one node has height `1`. A leaf has both child indices equal to `-1`.

Useful empty-subtree identities are `0` for count and sum, a number larger than every legal value for minimum, and a number smaller than every legal value for maximum. After the base case, combine the current node with both recursive results. The recursion depth is the tree height, so contest inputs must bound height safely below Python's recursion limit.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    root = int(tokens[1])
    val = []
    left = []
    right = []
    position = 2
    i = 0
    while i < n:
        val.append(int(tokens[position]))
        left.append(int(tokens[position + 1]))
        right.append(int(tokens[position + 2]))
        position = position + 3
        i = i + 1

    def score(node):
        if node == -1:
            return (0, 0, 0, 1000000001, -1000000001)
        left_score = score(left[node])
        right_score = score(right[node])
        height = 1 + max(left_score[0], right_score[0])
        leaves = left_score[1] + right_score[1]
        if left[node] == -1 and right[node] == -1:
            leaves = 1
        total = val[node] + left_score[2] + right_score[2]
        smallest = min(val[node], left_score[3], right_score[3])
        largest = max(val[node], left_score[4], right_score[4])
        return (height, leaves, total, smallest, largest)

    answer = score(root)
    return str(answer[0]) + " " + str(answer[1]) + " " + str(answer[2]) + " " + str(answer[3]) + " " + str(answer[4])

tree = "5 0 8 1 2 3 3 -1 10 -1 4 1 -1 -1 14 -1 -1"
assert solve(tree) == "3 2 36 1 14"
assert solve("1 0 -7 -1 -1") == "1 1 -7 -7 -7"

## Lesson 2 — Binary Search Trees

A **binary search tree**, or BST, adds an ordering rule. Every key in a node's left subtree is smaller than the node's key, and every key in its right subtree is larger. This unit's duplicate-key policy is: **ignore a key that is already present**. Therefore every stored key is distinct.

Search compares the target with the current key and recursively follows only one child. It takes time proportional to the tree height. A balanced tree can be fast, but an insertion order such as increasing values creates a right-only tree, so never assume a BST is balanced.

An in-order traversal of a BST lists its keys in sorted order. That is a useful fact, but graded traversal-output problems use pre-order or post-order so sorting the values cannot replace walking the given structure.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    root = int(tokens[1])
    target = int(tokens[2])
    val = []
    left = []
    right = []
    position = 3
    i = 0
    while i < n:
        val.append(int(tokens[position]))
        left.append(int(tokens[position + 1]))
        right.append(int(tokens[position + 2]))
        position = position + 3
        i = i + 1

    def contains(node):
        if node == -1:
            return False
        if val[node] == target:
            return True
        if target < val[node]:
            return contains(left[node])
        return contains(right[node])

    if contains(root):
        return "YES"
    return "NO"

tree = "5 0 14 8 1 2 3 3 -1 10 -1 4 1 -1 -1 14 -1 -1"
assert solve(tree) == "YES"
assert solve("5 0 13 8 1 2 3 3 -1 10 -1 4 1 -1 -1 14 -1 -1") == "NO"

## Build a BST by Recursive Insertion

Start with `root = -1`. To insert a key, reaching `node == -1` means the key belongs in a new node. Append its value and two `-1` children, then return its new index. Otherwise recursively insert into the left or right child and save the returned child index. If the key equals the current value, return the unchanged node to enforce the ignore-duplicates policy.

The insertion sequence controls the shape. Pre-order reveals that shape; sorting the input does not.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    val = []
    left = []
    right = []

    def insert(node, key):
        if node == -1:
            new_node = len(val)
            val.append(key)
            left.append(-1)
            right.append(-1)
            return new_node
        if key < val[node]:
            left[node] = insert(left[node], key)
        elif key > val[node]:
            right[node] = insert(right[node], key)
        return node

    def preorder(node):
        if node == -1:
            return ""
        text = str(val[node])
        left_text = preorder(left[node])
        right_text = preorder(right[node])
        if left_text != "":
            text = text + " " + left_text
        if right_text != "":
            text = text + " " + right_text
        return text

    root = -1
    i = 0
    while i < n:
        root = insert(root, int(tokens[i + 1]))
        i = i + 1
    return preorder(root)

assert solve("8 8 3 10 1 6 14 4 6") == "8 3 1 6 4 10 14"
assert solve("3 2 4 6") == "2 4 6"

## Validate Every BST Bound

Checking only a node against its parent is not enough. A value deep in the left subtree must still be smaller than the original root. Propagate an allowed open interval `(low, high)` through recursion:

- the current key must satisfy `key > low and key < high`;
- the left child receives `(low, key)`;
- the right child receives `(key, high)`.

The empty subtree is valid. Use separate comparisons instead of a chained comparison. Strict bounds match the no-duplicates policy.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    root = int(tokens[1])
    val = []
    left = []
    right = []
    position = 2
    i = 0
    while i < n:
        val.append(int(tokens[position]))
        left.append(int(tokens[position + 1]))
        right.append(int(tokens[position + 2]))
        position = position + 3
        i = i + 1

    def valid(node, low, high):
        if node == -1:
            return True
        if val[node] <= low or val[node] >= high:
            return False
        if not valid(left[node], low, val[node]):
            return False
        return valid(right[node], val[node], high)

    if valid(root, -1000000001, 1000000001):
        return "VALID"
    return "INVALID"

valid_tree = "5 0 8 1 2 3 3 -1 10 -1 4 1 -1 -1 14 -1 -1"
invalid_tree = "5 0 8 1 2 3 -1 3 10 -1 4 9 -1 -1 14 -1 -1"
assert solve(valid_tree) == "VALID"
assert solve(invalid_tree) == "INVALID"

## Binary Tree Checklist

Identify the root and the meaning of each parallel array. Base-case `node == -1` before every array access. Decide whether the node is processed before, between, or after its children. For aggregation, write the empty-subtree identity before the recursive case. For a BST, state the duplicate policy and never assume the height is balanced. For validation, carry all ancestor limits with propagated bounds.

## Submit the Solver

After `solve(data)` works, a contest submission can use this wrapper. It is marked `no-exec` because notebook checks call `solve` directly.

In [ ]:
import sys
print(solve(sys.stdin.read()))